In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models 
from torchvision import transforms
from torch.utils.data.dataset import random_split
from torch.utils.data import Dataset, Subset, ConcatDataset, DataLoader
from torchvision import datasets
import matplotlib.pyplot as plt
import matplotlib
from mpl_toolkits.axes_grid1 import ImageGrid
import pandas as pd
from torchview import draw_graph
try:
    from .algo import EmpBH, adaptiveEmpBH
except ImportError:
    from algo import EmpBH, adaptiveEmpBH
from medmnist import ChestMNIST, DermaMNIST, BreastMNIST
from procedure import AdaDetectERM

In [2]:
def get_fdp(ytrue, rejection_set):
    """
    """

    if rejection_set.size:
        fdp = np.sum(ytrue[rejection_set] == 0) / len(rejection_set)
        tdp = np.sum(ytrue[rejection_set] == 1) / np.sum(ytrue==1)
    else: 
        fdp=0
        tdp=0
    return fdp, tdp

### Load Data ####


MNIST

In [14]:
train_data = datasets.MNIST(
    root = 'data',
    train=True,                         
    transform = transforms.ToTensor(), 
    download = True,            
)




DERMAMNIST

In [ ]:
train_data = DermaMNIST(
    split='train',
    download=True,
    root='data',
    transform=transforms.ToTensor()
)
test_data = DermaMNIST(
    split='test',
    download=True,
    root='data',
    transform=transforms.ToTensor()
)
val_data = DermaMNIST(
    split='val',
    download=True,
    root='data',
    transform=transforms.ToTensor()
)

### Choose Classifier

In [15]:
class BinaryConvNet(nn.Module):
    def __init__(self, in_channels):
        super(BinaryConvNet, self).__init__()
        self.dim_target=1

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU())

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))

        self.layer3 = nn.Sequential(
            nn.Conv2d(16, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())
        
        self.layer4 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())

        self.layer5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))

        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [39]:
from torch import device


class NumpyDataset(Dataset):
    """Wrapper to convert numpy arrays to PyTorch Dataset"""
    def __init__(self, x, y):
        self.x = torch.from_numpy(x).float()
        self.y = torch.from_numpy(y).float()
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


class NNClassifier(object):
    """
    wrapper around NN model with fit() and predict_proba() functions
    """
    def __init__(self, model, batch_size=128, n_epochs=10, source_dataset=None):
        """
        Args:
            model: PyTorch model
            batch_size: batch size for training
            n_epochs: number of epochs
            source_dataset: original dataset to fetch images from (required when using indices)
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.loss_fn = nn.BCEWithLogitsLoss() 
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        self.source_dataset = source_dataset
    
        self.n_iter_no_change = 10
        self.tol = 1e-4

    def fit(self, train_dataset, val_dataset=None):
        # Convert numpy arrays or torch tensors to Dataset if needed
        if isinstance(train_dataset, (np.ndarray, torch.Tensor)):
            # train_dataset contains indices, val_dataset contains labels
            if self.source_dataset is None:
                raise ValueError("source_dataset must be provided when training with indices")
            
            # Convert to numpy if tensor
            if isinstance(train_dataset, torch.Tensor):
                train_dataset = train_dataset.cpu().numpy()
            
            indices = train_dataset.astype(int)
            labels = val_dataset if isinstance(val_dataset, np.ndarray) else val_dataset.cpu().numpy() if isinstance(val_dataset, torch.Tensor) else val_dataset
            labels = labels.astype(int)
            
            # Fetch actual images from source dataset
            images = []
            for idx in indices:
                img, _ = self.source_dataset[idx]
                images.append(img)
            images = torch.stack(images)
            labels_tensor = torch.from_numpy(labels).float()
            
            train_dataset = IndexedDataset(images, labels_tensor)
            val_dataset = None
        
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        
        if val_dataset is not None: 
            val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)

        # for early stopping
        best_val_acc = 0
        it_no_change = 0

        for _ in range(self.n_epochs):
            for x_batch, y_batch in train_loader:
                self.model.train()
                x_batch = x_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                y_pred = self.model(x_batch)
                y_batch = y_batch.unsqueeze(-1)
                loss = self.loss_fn(y_pred, y_batch)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()

            # must use early stopping to avoid overfitting
            if val_dataset is not None: 
                with torch.no_grad(): # after each epoch
                    val_acc = 0
                    for x_val, y_val in val_loader:
                        self.model.eval()
                        x_val = x_val.to(self.device)
                        y_val = y_val.to(self.device)
                        yhat = self.model(x_val)
                        yhat = yhat.softmax(dim=-1) # probabilities 
                        y_val = y_val.squeeze().long()
                        acc = torch.sum(torch.argmax(yhat, dim=1) == y_val).item()
                        val_acc += acc
                    val_acc /= len(val_loader.dataset)
                
                    if val_acc > best_val_acc + self.tol:
                        best_val_acc = val_acc
                        it_no_change = 0 # reset 
                    else:
                        it_no_change += 1
                        if it_no_change > self.n_iter_no_change: # break if too much 
                            break 

    def predict_proba(self, test_dataset):
        # Convert numpy arrays or torch tensors to Dataset if needed
        if isinstance(test_dataset, (np.ndarray, torch.Tensor)):
            if self.source_dataset is None:
                raise ValueError("source_dataset must be provided when predicting with indices")
            
            # Convert to numpy if tensor
            if isinstance(test_dataset, torch.Tensor):
                test_dataset = test_dataset.cpu().numpy()
            
            indices = test_dataset.astype(int)
            
            # Fetch actual images from source dataset
            images = []
            for idx in indices:
                img, _ = self.source_dataset[idx]
                images.append(img)
            images = torch.stack(images)
            
            test_dataset = IndexedDataset(images, torch.zeros(len(images)))
        
        test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

        with torch.no_grad():
            for x, y in test_loader: 
                self.model.eval()
                x = x.to(self.device)
                yhat_unnormed = self.model(x)
                yhat_logit = nn.functional.sigmoid(yhat_unnormed)
                yhat_numpy = yhat_logit.cpu().numpy()
                
                # Return 2D array with probabilities for both classes [prob_negative, prob_positive]
                prob_positive = yhat_numpy.flatten()
                prob_negative = 1.0 - prob_positive
                return np.column_stack([prob_negative, prob_positive])


### Prepare Data


In [27]:
class custom_subset(Dataset):
    r"""
    Subset of a dataset at specified indices.

    Arguments:
        dataset (Dataset): The whole Dataset
        indices (sequence): Indices in the whole set selected for subset
        labels(sequence) : targets as required for the indices. will be the same length as indices
    """
    def __init__(self, dataset, indices, labels=None):
        self.dataset = torch.utils.data.Subset(dataset, indices)
        self.targets = dataset.targets[indices] if labels is None else labels
    def __getitem__(self, idx):
        image = self.dataset[idx][0]
        target = self.targets[idx]
        return (image, target)

    def __len__(self):
        return len(self.targets)


class IndexedDataset(Dataset):
    """Dataset that stores preloaded images and labels for efficient training"""
    def __init__(self, images, labels):
        """
        Args:
            images: tensor of shape (N, C, H, W)
            labels: tensor of shape (N,)
        """
        self.images = images
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


In [28]:
inlr_labels=[4]
outlr_labels=[9]
print(train_data)
inlr = (torch.tensor(train_data.targets)[..., None] == torch.tensor(inlr_labels)).any(-1).nonzero(as_tuple=True)[0]
outlr = (torch.tensor(train_data.targets)[..., None] == torch.tensor(outlr_labels)).any(-1).nonzero(as_tuple=True)[0]

print(len(inlr), len(outlr))

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()
5842 5949


C:\Users\h4sor\AppData\Local\Temp\ipykernel_1524\4198790687.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inlr = (torch.tensor(train_data.targets)[..., None] == torch.tensor(inlr_labels)).any(-1).nonzero(as_tuple=True)[0]
C:\Users\h4sor\AppData\Local\Temp\ipykernel_1524\4198790687.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  outlr = (torch.tensor(train_data.targets)[..., None] == torch.tensor(outlr_labels)).any(-1).nonzero(as_tuple=True)[0]


#### Run Code

In [ ]:
#test sample 

m1 = 100
m0 = 900 
test1, test0 = outlr[:m1], inlr[:m0]
x = np.concatenate([test0, test1]) 

#NTS
n=5000
xnull = inlr[m0:m0+n]

#apply AdaDetect with ERM approach, with k=4000 (see notations of the paper)
level=0.05
# Use in_channels=1 for MNIST (grayscale)
proc = AdaDetectERM(scoring_fn = NNClassifier(BinaryConvNet(in_channels=1), n_epochs=20, source_dataset=train_data),
                                 split_size=4000/5000) 
proc.apply(x, level, xnull) #gives the rejection set 
